# Lab: Pandas techniques on the e-commerce dataset
Six exercises: descriptive statistics, groupby, crosstab/pivot, ranking, window operations, reshaping.

In [1]:
import pandas as pd

df = pd.read_csv("ecommerce_orders.csv")
df.head()

,Order Date,Region,Category,Product,Revenue
0,2022-01-06,West,Apparel,Jeans,58
1,2022-01-07,South,Home Goods,Coffee Maker,143
2,2022-01-15,West,Electronics,Tablet,614
3,2022-01-15,East,Apparel,Jacket,78
4,2022-01-19,East,Apparel,Jeans,78


## Check that the data is clean
The task says to start from a clean DataFrame, so I verify that first.

In [2]:
print(df.shape)
df.info()
print("Missing values:", df.isna().sum().sum(), "| Duplicate rows:", df.duplicated().sum())

(60, 5)
<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Order Date  60 non-null     str  
 1   Region      60 non-null     str  
 2   Category    60 non-null     str  
 3   Product     60 non-null     str  
 4   Revenue     60 non-null     int64
dtypes: int64(1), str(4)
memory usage: 2.5 KB
Missing values: 0 | Duplicate rows: 0


In [3]:
# Order Date is text, so convert it and sort by date (needed for pct_change and rolling later)
df["Order Date"] = pd.to_datetime(df["Order Date"])
df = df.sort_values("Order Date").reset_index(drop=True)
df.dtypes

Order Date    datetime64[us]
Region                   str
Category                 str
Product                  str
Revenue                int64
dtype: object

## Exercise 1: Descriptive statistics

In [4]:
df.describe()

,Order Date,Revenue
count,60,60.000000
mean,2022-04-09 19:36:00,276.316667
min,2022-01-06 00:00:00,15.000000
25%,2022-03-05 06:00:00,61.750000
50%,2022-04-04 12:00:00,97.500000
75%,2022-05-29 12:00:00,529.250000
max,2022-06-29 00:00:00,1058.000000
std,NaN,311.081792


In [5]:
total_revenue = df["Revenue"].sum()
avg_order_value = df["Revenue"].mean()
print(f"Total revenue:       {total_revenue}")
print(f"Average order value: {avg_order_value:.2f}")

Total revenue:       16579
Average order value: 276.32


In [6]:
# pct_change() compares each order with the previous order, which is noisy.
order_growth = df["Revenue"].pct_change()
print("Mean order-to-order change:", round(order_growth.mean() * 100, 1), "%")

# A more meaningful growth figure: month-over-month change in total revenue
monthly = df.set_index("Order Date")["Revenue"].resample("ME").sum()
monthly_growth = (monthly.pct_change() * 100).round(1)
pd.DataFrame({"Monthly revenue": monthly, "Growth %": monthly_growth})

Mean order-to-order change: 415.1 %


,Monthly revenue,Growth %
Order Date,,
2022-01-31,1020,NaN
2022-02-28,1469,44.0
2022-03-31,4984,239.3
2022-04-30,2235,-55.2
2022-05-31,2823,26.3
2022-06-30,4048,43.4


`pct_change()` on raw orders only compares one order with the one before it, so I also calculated month-over-month growth, which is the usual business meaning of growth percentage.

## Exercise 2: GroupBy and aggregation

In [7]:
region_category = df.groupby(["Region", "Category"])["Revenue"].sum().reset_index()
region_category

,Region,Category,Revenue
0,East,Apparel,430
1,East,Electronics,4656
2,East,Home Goods,307
3,South,Apparel,336
4,South,Electronics,3541
5,South,Home Goods,549
6,West,Apparel,465
7,West,Electronics,3824
8,West,Home Goods,2471


In [8]:
# Extra: several aggregations at once
df.groupby(["Region", "Category"])["Revenue"].agg(["sum", "mean", "count"]).round(1)

sum   mean  count
Region Category                       
East   Apparel       430   61.4      7
       Electronics  4656  582.0      8
       Home Goods    307   76.8      4
South  Apparel       336   48.0      7
       Electronics  3541  590.2      6
       Home Goods    549   91.5      6
West   Apparel       465   66.4      7
       Electronics  3824  546.3      7
       Home Goods   2471  308.9      8

## Exercise 3: Crosstab and pivot table

In [9]:
pd.crosstab(df["Region"], df["Category"])

Category,Apparel,Electronics,Home Goods
Region,,,
East,7,8,4
South,7,6,6
West,7,7,8


In [10]:
pd.pivot_table(df, values="Revenue", index="Region", columns="Category",
               aggfunc="sum", fill_value=0)

Category,Apparel,Electronics,Home Goods
Region,,,
East,430,4656,307
South,336,3541,549
West,465,3824,2471


## Exercise 4: Ranking and sorting
The slide's snippet ranks individual *orders*. To find the top *products*, first total revenue per Region + Product, then rank inside each region.

In [11]:
product_revenue = df.groupby(["Region", "Product"])["Revenue"].sum().reset_index()

product_revenue["RevenueRank"] = (
    product_revenue.groupby("Region")["Revenue"].rank(method="dense", ascending=False)
)
top_3_per_region = (
    product_revenue[product_revenue["RevenueRank"] <= 3]
    .sort_values(["Region", "RevenueRank"])
)
top_3_per_region

,Region,Product,Revenue,RevenueRank
6,East,Smartphones,2641,1.0
8,East,Tablet,1305,2.0
5,East,Laptop,710,3.0
14,South,Laptop,1981,1.0
15,South,Smartphones,1324,2.0
10,South,Coffee Maker,420,3.0
20,West,Couch,2039,1.0
25,West,Laptop,1903,2.0
26,West,Smartphones,1090,3.0


## Exercise 5: Window operations

In [12]:
df_ts = df.set_index("Order Date")   # copy, so df stays usable for Exercise 6
df_ts.index.dtype

dtype('<M8[us]')

In [13]:
# Monthly totals first (the tip), then a 3-month rolling average
monthly = df_ts["Revenue"].resample("ME").sum().to_frame("MonthlyRevenue")
monthly["rolling_avg"] = monthly["MonthlyRevenue"].rolling(window=3).mean()
monthly

,MonthlyRevenue,rolling_avg
Order Date,,
2022-01-31,1020,NaN
2022-02-28,1469,NaN
2022-03-31,4984,2491.000000
2022-04-30,2235,2896.000000
2022-05-31,2823,3347.333333
2022-06-30,4048,3035.333333


The first two months are `NaN` because a 3-month window needs three months of data. Rolling directly on the order rows would average 3 *orders*, not 3 months.

## Exercise 6: Data reshaping

In [14]:
# The slide assumes a wide table already exists, so I build one: Region x Quarter
df["Quarter"] = "Q" + df["Order Date"].dt.quarter.astype(str) + "_Sales"
df_wide = (
    df.pivot_table(index="Region", columns="Quarter", values="Revenue", aggfunc="sum", fill_value=0)
      .reset_index()
)
df_wide.columns.name = None
df_wide

,Region,Q1_Sales,Q2_Sales
0,East,2857,2536
1,South,2030,2396
2,West,2586,4174


In [15]:
# Wide to long
quarter_cols = [c for c in df_wide.columns if c != "Region"]
df_long = pd.melt(df_wide, id_vars=["Region"], value_vars=quarter_cols,
                  var_name="Quarter", value_name="Sales")
df_long

,Region,Quarter,Sales
0,East,Q1_Sales,2857
1,South,Q1_Sales,2030
2,West,Q1_Sales,2586
3,East,Q2_Sales,2536
4,South,Q2_Sales,2396
5,West,Q2_Sales,4174


In [16]:
# Long to wide
df_wide_again = df_long.pivot(index="Region", columns="Quarter", values="Sales")
df_wide_again

Quarter,Q1_Sales,Q2_Sales
Region,,
East,2857,2536
South,2030,2396
West,2586,4174


In [17]:
# Round trip check
print(df_wide_again.reset_index().rename_axis(columns=None).equals(df_wide))

True


The data covers January to June 2022, so there are only `Q1_Sales` and `Q2_Sales`. The slide's `Q3_Sales` column does not exist in this dataset.

## Summary
Cleaned/checked the data, computed descriptive statistics, grouped and pivoted revenue by Region and Category, ranked the top 3 products per region, calculated a 3-month rolling average on monthly totals, and reshaped between wide and long formats with `melt` and `pivot`.